<a href="https://colab.research.google.com/github/asaveraasad-data/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os
import subprocess

REPO_URL = "https://github.com/asaveraasad-data/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship/flyrank-ml-internship


In [17]:
import os

print(os.getcwd())
print(os.listdir())

/content/flyrank-ml-internship/flyrank-ml-internship
['LICENSE', 'notebooks', 'skills', '.github', 'CLAUDE.md', 'data', '.git', 'submission', 'AGENTS.md', 'docs', 'requirements.txt', 'outputs', 'README.md', 'work', 'DATA_USE.md', 'GUIDE.md', '.gitignore', 'SETUP.md', 'scripts']


## 1. Method choice and why

### Selected Method

I selected **Logistic Regression** as the primary model for this task.

My goal is to identify content pages that should be prioritized for content refresh. This is a binary classification problem because each page can either be classified as needing review or not needing review.

Logistic Regression is a good starting point because it is simple, interpretable, and provides prediction probabilities that can also be used for ranking pages. It serves as a transparent benchmark before considering more complex models such as Decision Trees or Random Forests.

The model will be compared against the Week 4 rule-based baseline using the same dataset, the same train/test split, and the same evaluation metrics.

In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

RANDOM_STATE = 42

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Random seed:", RANDOM_STATE)
print("Dataset shape:", df.shape)

df.head()

Random seed: 42
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [19]:
# Create the target variable

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nPercentage:")
print(
    (df["is_declining_label"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Percentage:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


### Split Design

I used **GroupShuffleSplit** with `client_id` as the grouping variable.

This ensures that content from the same client never appears in both the training and testing sets. Grouped validation is more honest because it evaluates how well the model generalizes to completely unseen clients rather than memorizing client-specific patterns.

Training Set:
- 23,837 rows
- 25 unique clients

Testing Set:
- 6,163 rows
- 7 unique clients

Shared clients between train and test: **0**

In [20]:
# Honest train/test split grouped by client

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print()
print("Unique train clients:", train_df["client_id"].nunique())
print("Unique test clients:", test_df["client_id"].nunique())

print()
print(
    "Clients shared:",
    len(
        set(train_df["client_id"]).intersection(
            set(test_df["client_id"])
        )
    )
)

Train shape: (23837, 45)
Test shape: (6163, 45)

Unique train clients: 25
Unique test clients: 7

Clients shared: 0


### Model Performance

The Logistic Regression model was trained using five leakage-safe features:

- days_since_last_update
- impressions_90d
- ctr
- avg_position
- content_age_days

The same grouped train/test split was used for evaluation.

Results:

| Metric | Logistic Regression |
|---------|--------------------:|
| Accuracy | 0.5377 |
| Precision | 0.5433 |
| Recall | 0.5973 |
| F1 Score | 0.5691 |

This model provides an interpretable baseline that can later be compared with more complex models such as Decision Trees or Random Forests.

In [21]:
# Features used for the model

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

target = "is_declining_label"

X_train = train_df[features]
X_test = test_df[features]

y_train = train_df[target]
y_test = test_df[target]

print("Features:")
print(features)

print("\nMissing values:")
print(X_train.isnull().sum())

Features:
['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days']

Missing values:
days_since_last_update    0
impressions_90d           0
ctr                       0
avg_position              0
content_age_days          0
dtype: int64


### Error Analysis

The model correctly identifies many declining pages but also produces both false positives and false negatives.

The confusion matrix shows that the classes overlap considerably, indicating that predicting content decline from only five features is challenging.

Feature importance (based on Logistic Regression coefficients) shows that:

1. CTR has the strongest influence on predictions.
2. Days since last update also contributes positively.
3. Content age has a smaller effect.
4. Average position and impressions contribute very little.

This suggests that user engagement signals are more informative than raw visibility metrics for predicting declining content.

In [22]:
# Train Logistic Regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Model trained successfully!")

Model trained successfully!


**Evaluation**

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

Accuracy : 0.5377
Precision: 0.5433
Recall   : 0.5973
F1 Score : 0.5691

Confusion Matrix
[[1433 1581]
 [1268 1881]]


**Feature Interpretation**

In [24]:
# Logistic Regression coefficients

coef = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
})

coef["Absolute"] = coef["Coefficient"].abs()
coef = coef.sort_values("Absolute", ascending=False)

coef

,Feature,Coefficient,Absolute
2,ctr,-0.055671,0.055671
0,days_since_last_update,0.005664,0.005664
4,content_age_days,-0.003083,0.003083
3,avg_position,-0.000163,0.000163
1,impressions_90d,-0.000004,0.000004


## Self-check

Before you submit, confirm each line honestly:

✅Every section above is filled — markdown thinking AND the code that backs it

✅The notebook runs top to bottom with no errors (Runtime → Run all)

✅No client names, URLs, or private queries anywhere

✅My claims use careful words: observed, measured, directional, decision-support

✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.